# Fraud Detection Pipeline — Stage 2: Model Development

**Author:** [Partner Name] — Model Development  
**Dataset:** [Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud) — ULB Machine Learning Group  
**Collaborator:** Muhammad Patel — Data Engineering (see `data_pipeline.ipynb`)

---

## Overview

This notebook covers **Stage 2** of a two-stage collaborative fraud detection pipeline. The clean, balanced datasets produced in `data_pipeline.ipynb` are loaded here and used to train, evaluate, and compare **6 classifiers**.

The central challenge is not just maximising accuracy — it is finding the best **precision/recall trade-off** for a banking environment where:
- A **False Negative** (missed fraud) has a direct financial cost to the bank
- A **False Positive** (legitimate transaction flagged) erodes customer trust

### Contributions
1. **Multi-model training** — 6 classifiers trained on the balanced dataset
2. **Performance comparison** — ranked by F1-score, Precision, and Recall
3. **Production model selection** — trade-off analysis between KNN and Random Forest
4. **Model serialisation** — saving the production model for API deployment

---

## 1. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import pathlib
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn import svm
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_curve, roc_curve, auc
)

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

# Output directories
pathlib.Path('images').mkdir(exist_ok=True)
pathlib.Path('models').mkdir(exist_ok=True)

## 2. Load Processed Datasets

We load the outputs from `data_pipeline.ipynb`:
- **`X_train_balanced` / `y_train_balanced`** — the 50/50 balanced training set (post-RUS)
- **`X_val` / `y_val`** — the imbalanced validation set for model selection
- **`X_test` / `y_test`** — the imbalanced test set, held out until final evaluation only

> ⚠️ The test set is not touched until Section 6 (Final Evaluation). Using it earlier would constitute data leakage.

In [ ]:
data_dir = pathlib.Path('ML Data')

X_train = pd.read_csv(data_dir / 'X_train_balanced.csv')
y_train = pd.read_csv(data_dir / 'y_train_balanced.csv').squeeze()
X_val   = pd.read_csv(data_dir / 'X_val.csv')
y_val   = pd.read_csv(data_dir / 'y_val.csv').squeeze()
X_test  = pd.read_csv(data_dir / 'X_test.csv')
y_test  = pd.read_csv(data_dir / 'y_test.csv').squeeze()

print(f"Training set (balanced): {X_train.shape} | Fraud rate: {y_train.mean()*100:.1f}%")
print(f"Validation set:          {X_val.shape}   | Fraud rate: {y_val.mean()*100:.2f}%")
print(f"Test set:                {X_test.shape}   | Fraud rate: {y_test.mean()*100:.2f}%")

## 3. Train & Evaluate All Classifiers

Six classifiers are trained on the balanced training set and evaluated on the **imbalanced validation set** — reflecting real-world conditions where 99.83% of transactions are legitimate.

Metrics collected per model:
- **Precision** — of all transactions flagged as fraud, how many actually were?
- **Recall** — of all actual fraud cases, how many did we catch?
- **F1-Score** — harmonic mean of Precision and Recall; the primary ranking metric
- **Accuracy** — included for completeness but not used for model selection (misleading on imbalanced data)

In [ ]:
models = {
    "Logistic Regression":        LogisticRegression(random_state=0, max_iter=1000),
    "Random Forest":               RandomForestClassifier(random_state=0),
    "Gradient Boosting":           GradientBoostingClassifier(random_state=0),
    "Naive Bayes":                 GaussianNB(),
    "K-Nearest Neighbours":        KNeighborsClassifier(),
    "Support Vector Machine":      svm.SVC(probability=True, random_state=0)
}

results = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train, np.ravel(y_train))
    y_pred = model.predict(X_val)
    trained_models[name] = model

    results.append({
        'Model':     name,
        'Accuracy':  accuracy_score(y_val, y_pred),
        'Precision': precision_score(y_val, y_pred),
        'Recall':    recall_score(y_val, y_pred),
        'F1 Score':  f1_score(y_val, y_pred)
    })
    print(f"✓ {name} trained")

results_df = pd.DataFrame(results).sort_values('F1 Score', ascending=False).reset_index(drop=True)
print("\n", results_df.round(4).to_string(index=False))

## 4. Model Comparison Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# F1 Score bar chart
colors = ['#E84855' if m == 'Random Forest' else '#2E86AB' for m in results_df['Model']]
axes[0].barh(results_df['Model'], results_df['F1 Score'], color=colors, edgecolor='white')
axes[0].set_title('F1 Score by Model (Validation Set)', fontweight='bold')
axes[0].set_xlabel('F1 Score')
axes[0].set_xlim(0, 1)
for i, v in enumerate(results_df['F1 Score']):
    axes[0].text(v + 0.005, i, f'{v:.4f}', va='center', fontsize=9)
axes[0].invert_yaxis()

# Precision vs Recall scatter
for _, row in results_df.iterrows():
    color = '#E84855' if row['Model'] == 'Random Forest' else '#2E86AB'
    axes[1].scatter(row['Recall'], row['Precision'], s=120, color=color, zorder=5)
    axes[1].annotate(row['Model'], (row['Recall'], row['Precision']),
                     textcoords='offset points', xytext=(6, 4), fontsize=8)

axes[1].set_title('Precision vs Recall Trade-off (Validation Set)', fontweight='bold')
axes[1].set_xlabel('Recall (Fraud Caught)')
axes[1].set_ylabel('Precision (Flagged = Actually Fraud)')
axes[1].set_xlim(0, 1.1)
axes[1].set_ylim(0, 1.1)

plt.tight_layout()
plt.savefig('images/model_comparison.png', bbox_inches='tight')
plt.show()

## 5. Production Model Selection — KNN vs Random Forest

### The Conflict

K-Nearest Neighbours achieved the highest raw F1-score on the validation set. However, **raw performance is not the only criterion for a production model** in a real-time banking environment.

### The Decision: Random Forest

| Criterion | KNN | Random Forest | Winner |
|---|---|---|---|
| F1 Score | Higher | Slightly lower | KNN |
| Inference speed | O(n) — scans all training points per prediction | O(log n) — traverses trees | **Random Forest** |
| High-dimensional data | Sensitive to the curse of dimensionality (30 features) | Handles well via feature subsampling | **Random Forest** |
| Noise sensitivity | High — one noisy neighbour skews prediction | Low — ensemble averaging reduces noise | **Random Forest** |
| Memory at inference | Requires full training set in memory | Only the trained tree structure | **Random Forest** |

> **Key insight:** A fraud detection model must make decisions in milliseconds at transaction time. KNN's O(n) inference cost makes it impractical at scale — a marginal F1 gain does not justify the infrastructure cost.

In [ ]:
# Side-by-side confusion matrices: KNN vs Random Forest
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, name in zip(axes, ['K-Nearest Neighbours', 'Random Forest']):
    y_pred = trained_models[name].predict(X_val)
    cm = confusion_matrix(y_val, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Legitimate', 'Fraud'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    f1 = f1_score(y_val, y_pred)
    prec = precision_score(y_val, y_pred)
    rec = recall_score(y_val, y_pred)
    verdict = '✓ SELECTED' if name == 'Random Forest' else '✗ Rejected'
    ax.set_title(f'{name}\nF1: {f1:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f}\n{verdict}',
                 fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig('images/knn_vs_rf_confusion.png', bbox_inches='tight')
plt.show()

## 6. Precision-Recall Curve — Random Forest

The Precision-Recall curve shows how the model's precision and recall change as the **decision threshold** shifts. In a banking context, a risk team might choose to tighten the threshold (higher precision, fewer false positives) or loosen it (higher recall, fewer missed frauds) depending on business policy.

In [ ]:
rf_model = trained_models['Random Forest']
y_scores = rf_model.predict_proba(X_val)[:, 1]

precision_vals, recall_vals, thresholds = precision_recall_curve(y_val, y_scores)
pr_auc = auc(recall_vals, precision_vals)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(recall_vals, precision_vals, color='#E84855', lw=2, label=f'Random Forest (AUC = {pr_auc:.4f})')
ax.axhline(y=y_val.mean(), color='grey', linestyle='--', label=f'Baseline (fraud rate = {y_val.mean():.4f})')
ax.set_xlabel('Recall (Fraud Caught)', fontsize=11)
ax.set_ylabel('Precision (Flagged = Actually Fraud)', fontsize=11)
ax.set_title('Precision-Recall Curve — Random Forest (Validation Set)', fontweight='bold')
ax.legend()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig('images/precision_recall_curve.png', bbox_inches='tight')
plt.show()

print(f"PR-AUC: {pr_auc:.4f}")

## 7. Final Evaluation on Test Set

> ⚠️ **This cell is run only once.** The test set has been untouched throughout model development. Evaluating here gives an unbiased estimate of real-world performance.

In [ ]:
y_test_pred = rf_model.predict(X_test)

print("=" * 55)
print("FINAL EVALUATION — Random Forest on Held-Out Test Set")
print("=" * 55)
print(f"Precision: {precision_score(y_test, y_test_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_test_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_test_pred):.4f}")
print(f"Accuracy:  {accuracy_score(y_test, y_test_pred):.4f}")
print()
print("Classification Report:")
print("─" * 55)
print(classification_report(y_test, y_test_pred, target_names=['Legitimate', 'Fraud']))

fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_test_pred)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Legitimate', 'Fraud']).plot(
    ax=ax, colorbar=False, cmap='Blues'
)
ax.set_title('Random Forest — Final Test Set Confusion Matrix', fontweight='bold')
plt.tight_layout()
plt.savefig('images/final_confusion_matrix.png', bbox_inches='tight')
plt.show()

## 8. Save Production Model

In [ ]:
model_path = pathlib.Path('models') / 'fraud_det_model.pkl'
joblib.dump(rf_model, model_path)
print(f"Production model saved to: {model_path}")
print(f"Model type: {type(rf_model).__name__}")

## 9. Results Summary

| Model | Precision | Recall | F1 Score | Verdict |
|---|---|---|---|---|
| **Random Forest** | **0.9310** | **0.7297** | **0.8182** | ✓ Selected — best balance for production |
| K-Nearest Neighbours | 0.9333 | 0.7568 | 0.8358 | Rejected — prohibitive inference cost at scale |
| Gradient Boosting | — | — | — | — |
| Logistic Regression | 0.8814 | 0.7027 | 0.7820 | Baseline (set by data pipeline team) |
| Support Vector Machine | — | — | — | — |
| Naive Bayes | 0.0613 | 0.8378 | 0.1143 | Rejected — 6% precision floods customers with false alerts |

> The production model is serialised to `models/fraud_det_model.pkl` and served via the REST API in `src/fraud_det_api.py`.

---
*Dataset source: [Kaggle — Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)*